In [1]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.60.0 which is incompatible.


In [11]:
import os
from google import genai
from google.colab import userdata

os.environ["Gemini_Api_key"] = userdata.get("Gemini_Api_key")

client = genai.Client(
    api_key=os.environ["Gemini_Api_key"]
)

In [3]:
required_fields = [
    "customer_name",
    "order_id",
    "issue_type",
    "requested_action",
    "priority"
]

print(required_fields)

['customer_name', 'order_id', 'issue_type', 'requested_action', 'priority']


In [4]:
example_document = """
Hi Support, I am Anitha.
My order ORD10245 arrived damaged.
Please arrange a replacement.
This is urgent.
"""

example_output = {
    "customer_name": "Anitha",
    "order_id": "ORD10245",
    "issue_type": "Product Issue",
    "requested_action": "Replacement",
    "priority": "High"
}

print("Example Document:")
print(example_document)

print("\nExample Output:")
print(example_output)

Example Document:

Hi Support, I am Anitha.
My order ORD10245 arrived damaged.
Please arrange a replacement.
This is urgent.


Example Output:
{'customer_name': 'Anitha', 'order_id': 'ORD10245', 'issue_type': 'Product Issue', 'requested_action': 'Replacement', 'priority': 'High'}


In [5]:
import json

prompt = f"""
You are an information extraction assistant.

Extract the following fields from the customer document:

{", ".join(required_fields)}

ONE-SHOT EXAMPLE:

Customer Document:
{example_document}

Expected JSON:
{json.dumps(example_output, indent=2)}

RULES:

- Return valid JSON only.
- Use exactly the required field names.
- Do not add extra fields.
- Do not invent information.
- If a field is missing, return null.
- Do not provide explanations.

Now extract the information from the new customer document.
"""

print(prompt)


You are an information extraction assistant.

Extract the following fields from the customer document:

customer_name, order_id, issue_type, requested_action, priority

ONE-SHOT EXAMPLE:

Customer Document:

Hi Support, I am Anitha.
My order ORD10245 arrived damaged.
Please arrange a replacement.
This is urgent.


Expected JSON:
{
  "customer_name": "Anitha",
  "order_id": "ORD10245",
  "issue_type": "Product Issue",
  "requested_action": "Replacement",
  "priority": "High"
}

RULES:

- Return valid JSON only.
- Use exactly the required field names.
- Do not add extra fields.
- Do not invent information.
- If a field is missing, return null.
- Do not provide explanations.

Now extract the information from the new customer document.



In [6]:
customer_document = """
Hi Support,

My name is Ravi.
My order ORD20456 arrived damaged.
Please arrange a replacement.
This is urgent.

Thank you.
"""

print(customer_document)


Hi Support,

My name is Ravi.
My order ORD20456 arrived damaged.
Please arrange a replacement.
This is urgent.

Thank you.



In [12]:
full_prompt = f"""
{prompt}

NEW CUSTOMER DOCUMENT:

{customer_document}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=full_prompt
)

print(response.text)

{
  "customer_name": "Ravi",
  "order_id": "ORD20456",
  "issue_type": "Product Issue",
  "requested_action": "Replacement",
  "priority": "High"
}


In [13]:
import json

result_text = response.text.strip()

print("Raw Gemini Output:")
print(result_text)

Raw Gemini Output:
{
  "customer_name": "Ravi",
  "order_id": "ORD20456",
  "issue_type": "Product Issue",
  "requested_action": "Replacement",
  "priority": "High"
}


In [15]:
data = json.loads(result_text)

print(data)

{'customer_name': 'Ravi', 'order_id': 'ORD20456', 'issue_type': 'Product Issue', 'requested_action': 'Replacement', 'priority': 'High'}


In [16]:
missing_fields = [
    field
    for field in required_fields
    if field not in data
]

if missing_fields:
    print("❌ Missing fields:")
    print(missing_fields)

else:
    print("✅ All required fields are present.")

✅ All required fields are present.


In [17]:
final_output = json.dumps(
    data,
    indent=4,
    ensure_ascii=False
)

print(final_output)

{
    "customer_name": "Ravi",
    "order_id": "ORD20456",
    "issue_type": "Product Issue",
    "requested_action": "Replacement",
    "priority": "High"
}


In [18]:
missing_field_document = """
Hi Support,

My name is Priya.
My order ORD30987 arrived damaged.

Please arrange a replacement.

Thank you.
"""

print(missing_field_document)


Hi Support,

My name is Priya.
My order ORD30987 arrived damaged.

Please arrange a replacement.

Thank you.



In [19]:
missing_prompt = f"""
{prompt}

NEW CUSTOMER DOCUMENT:

{missing_field_document}
"""

missing_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=missing_prompt
)

print(missing_response.text)

{
  "customer_name": "Priya",
  "order_id": "ORD30987",
  "issue_type": "Product Issue",
  "requested_action": "Replacement",
  "priority": null
}


In [20]:
missing_result_text = missing_response.text.strip()

missing_data = json.loads(missing_result_text)

missing_fields = [
    field
    for field in required_fields
    if field not in missing_data
]

if missing_fields:
    print("❌ Required fields missing from JSON:")
    print(missing_fields)

else:
    print("✅ All required fields are present.")

    print("\nFinal Missing-Field Test Output:")
    print(
        json.dumps(
            missing_data,
            indent=4,
            ensure_ascii=False
        )
    )

✅ All required fields are present.

Final Missing-Field Test Output:
{
    "customer_name": "Priya",
    "order_id": "ORD30987",
    "issue_type": "Product Issue",
    "requested_action": "Replacement",
    "priority": null
}


In [21]:
extra_fields = [
    field
    for field in data
    if field not in required_fields
]

if extra_fields:
    print("❌ Extra fields found:")
    print(extra_fields)

else:
    print("✅ No extra fields found.")

✅ No extra fields found.


In [22]:
improved_prompt = f"""
You are a structured information extraction assistant.

Your task is to extract information from a customer-support
document and return ONLY a valid JSON object.

REQUIRED FIELDS:

{", ".join(required_fields)}

ONE-SHOT EXAMPLE:

Customer Document:
{example_document}

Expected JSON:
{json.dumps(example_output, indent=4)}

STRICT RULES:

1. Return ONLY JSON.
2. Do not use Markdown.
3. Do not add explanations.
4. Use exactly the five required field names.
5. Do not add extra fields.
6. Do not invent or guess information.
7. Extract information only when it is explicitly present
   in the document.
8. If a required field is not available, use null.
9. Keep the values concise and relevant.
10. Return the same JSON structure as the example.

Now extract information from the following document:

{customer_document}
"""

print(improved_prompt)


You are a structured information extraction assistant.

Your task is to extract information from a customer-support
document and return ONLY a valid JSON object.

REQUIRED FIELDS:

customer_name, order_id, issue_type, requested_action, priority

ONE-SHOT EXAMPLE:

Customer Document:

Hi Support, I am Anitha.
My order ORD10245 arrived damaged.
Please arrange a replacement.
This is urgent.


Expected JSON:
{
    "customer_name": "Anitha",
    "order_id": "ORD10245",
    "issue_type": "Product Issue",
    "requested_action": "Replacement",
    "priority": "High"
}

STRICT RULES:

1. Return ONLY JSON.
2. Do not use Markdown.
3. Do not add explanations.
4. Use exactly the five required field names.
5. Do not add extra fields.
6. Do not invent or guess information.
7. Extract information only when it is explicitly present
   in the document.
8. If a required field is not available, use null.
9. Keep the values concise and relevant.
10. Return the same JSON structure as the example.

Now ext

In [23]:
improved_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=improved_prompt
)

print(improved_response.text)

{
    "customer_name": "Ravi",
    "order_id": "ORD20456",
    "issue_type": "Product Issue",
    "requested_action": "Replacement",
    "priority": "High"
}


In [24]:
improved_result_text = improved_response.text.strip()

# Convert Gemini output into Python dictionary
improved_data = json.loads(improved_result_text)

# Check missing fields
missing_fields = [
    field
    for field in required_fields
    if field not in improved_data
]

# Check extra fields
extra_fields = [
    field
    for field in improved_data
    if field not in required_fields
]


if missing_fields:

    print("❌ Missing fields:")
    print(missing_fields)

elif extra_fields:

    print("❌ Extra fields:")
    print(extra_fields)

else:

    print("✅ JSON validation successful.")

    print("\nFinal Output:")

    print(
        json.dumps(
            improved_data,
            indent=4,
            ensure_ascii=False
        )
    )

✅ JSON validation successful.

Final Output:
{
    "customer_name": "Ravi",
    "order_id": "ORD20456",
    "issue_type": "Product Issue",
    "requested_action": "Replacement",
    "priority": "High"
}


In [25]:
test_document = """
Hello Support Team,

I am Kumar.
My order number is ORD50123.
The package has not arrived yet.

Please check the delivery status.
This issue is urgent.

Thank you.
"""

print(test_document)


Hello Support Team,

I am Kumar.
My order number is ORD50123.
The package has not arrived yet.

Please check the delivery status.
This issue is urgent.

Thank you.



In [26]:
test_prompt = f"""
You are a structured information extraction assistant.

Your task is to extract information from a customer-support
document and return ONLY a valid JSON object.

REQUIRED FIELDS:

{", ".join(required_fields)}

ONE-SHOT EXAMPLE:

Customer Document:
{example_document}

Expected JSON:
{json.dumps(example_output, indent=4)}

STRICT RULES:

1. Return ONLY JSON.
2. Do not use Markdown.
3. Do not add explanations.
4. Use exactly the five required field names.
5. Do not add extra fields.
6. Do not invent or guess information.
7. Extract information only when it is explicitly present.
8. If a required field is not available, use null.
9. Return the same JSON structure as the example.

NEW CUSTOMER DOCUMENT:

{test_document}
"""

In [27]:
test_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=test_prompt
)

print(test_response.text)

{
    "customer_name": "Kumar",
    "order_id": "ORD50123",
    "issue_type": "Shipping Issue",
    "requested_action": "Check delivery status",
    "priority": "High"
}


In [28]:
test_result_text = test_response.text.strip()

# Convert Gemini output to Python dictionary
test_data = json.loads(test_result_text)

# Check missing fields
missing_fields = [
    field
    for field in required_fields
    if field not in test_data
]

# Check extra fields
extra_fields = [
    field
    for field in test_data
    if field not in required_fields
]


if missing_fields:

    print("❌ Missing fields:")
    print(missing_fields)

elif extra_fields:

    print("❌ Extra fields:")
    print(extra_fields)

else:

    print("✅ Final test validation successful.")

    print("\nExtracted Information:")

    print(
        json.dumps(
            test_data,
            indent=4,
            ensure_ascii=False
        )
    )

✅ Final test validation successful.

Extracted Information:
{
    "customer_name": "Kumar",
    "order_id": "ORD50123",
    "issue_type": "Shipping Issue",
    "requested_action": "Check delivery status",
    "priority": "High"
}


STEP 1  → Install packages
STEP 2  → Gemini client
STEP 3  → Required schema
STEP 4  → One-shot example
STEP 5  → Extraction prompt
STEP 6  → Customer document
STEP 7  → Gemini extraction
STEP 9  → JSON conversion
STEP 10 → Required-field validation
STEP 11 → Final JSON
STEP 12 → Missing-field test
STEP 13 → Missing-field validation
STEP 14 → Extra-field check
STEP 15 → Improved prompt
STEP 16 → Improved prompt test
STEP 17 → Improved output validation
STEP 18 → New test document
STEP 19 → Final validation

In [29]:
!pip install -q -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.1 MB/s eta 0:00:00


In [30]:
import gradio as gr
import json


def extract_information(document):

    if not document.strip():
        return "Please enter a customer document."

    final_prompt = f"""
You are a structured information extraction assistant.

Extract exactly these fields:

{", ".join(required_fields)}

ONE-SHOT EXAMPLE:

Customer Document:
{example_document}

Expected JSON:
{json.dumps(example_output, indent=4)}

STRICT RULES:

1. Return ONLY valid JSON.
2. Do not use Markdown.
3. Do not add explanations.
4. Use exactly the five required fields.
5. Do not add extra fields.
6. Do not invent or guess information.
7. If information is missing, return null.

NEW CUSTOMER DOCUMENT:

{document}
"""

    try:

        # Gemini call
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=final_prompt
        )

        result = response.text.strip()

        # Remove ```json if Gemini adds it
        if result.startswith("```json"):
            result = result[7:]

        if result.startswith("```"):
            result = result[3:]

        if result.endswith("```"):
            result = result[:-3]

        result = result.strip()

        # Convert to JSON
        data = json.loads(result)

        # Check missing fields
        missing_fields = [
            field
            for field in required_fields
            if field not in data
        ]

        # Check extra fields
        extra_fields = [
            field
            for field in data
            if field not in required_fields
        ]

        if missing_fields:
            return json.dumps(
                {
                    "error": "Missing required fields",
                    "missing_fields": missing_fields
                },
                indent=4
            )

        if extra_fields:
            return json.dumps(
                {
                    "error": "Extra fields found",
                    "extra_fields": extra_fields
                },
                indent=4
            )

        # Final output
        return json.dumps(
            data,
            indent=4,
            ensure_ascii=False
        )

    except json.JSONDecodeError:
        return "❌ Gemini returned invalid JSON:\n\n" + result

    except Exception as e:
        return "❌ Error:\n\n" + str(e)


# =========================
# GRADIO UI
# =========================

demo = gr.Interface(
    fn=extract_information,

    inputs=gr.Textbox(
        label="📄 Customer Document",
        placeholder="Enter customer support document...",
        lines=10
    ),

    outputs=gr.Code(
        label="📋 Extracted Information",
        language="json"
    ),

    title="📄 Customer Document Information Extractor",

    description=(
        "One-Shot Prompting based structured "
        "information extraction using Gemini"
    ),

    examples=[
        [
            """Hi Support,
I am Ravi.
My order ORD20456 arrived damaged.
Please arrange a replacement.
This is urgent."""
        ],
        [
            """Hello Support,
My name is Priya.
My order ORD30987 has a payment issue.
I would like a refund."""
        ]
    ]
)


# Launch Gradio
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ada8f7fecb391510d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
